In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Statistical Analysis

### Task 1 — T-test + Effect Size: BMI vs Diabetes (20 min)

1) Question: Is the difference in mean BMI between diabetic and non-diabetic patients statistically significant, and how large is the difference?

2) H0: There is no significant difference in mean BMI between diabetic and non-diabetic patients (mu_diabetic = mu_non-diabetic)

   H1: There is a significant difference in mean BMI between diabetic and non-diabetic patients (mu_diabetic != mu_non-diabetic)

In [ ]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

In [ ]:
bct = pd.crosstab(df["_BMI5"],df["Diabetes_binary"],normalize='index')
print(bct)

In [ ]:
df.groupby("Diabetes_binary")["_BMI5"].count()

In [ ]:
BMI_nondiabetic = df[df["Diabetes_binary"]==0]['_BMI5'].reset_index(drop=True)
BMI_diabetic = df[df["Diabetes_binary"]==1]['_BMI5'].reset_index(drop=True)

In [ ]:
print(stats.ttest_ind(BMI_diabetic,BMI_nondiabetic))

In [ ]:
bmi_d = BMI_diabetic/100
bmi_nond = BMI_nondiabetic/100

In [ ]:
print(bmi_d.std())
print(bmi_nond.std())

In [ ]:
n1 = len(BMI_diabetic)
n2 = len(BMI_nondiabetic)
sd1 = bmi_d.std()
sd2 = bmi_nond.std()

pooled_std = np.sqrt(((n1-1)*sd1**2+(n2-1)*sd2**2)/ (n1+n2-2))
effect_size = (bmi_d.mean()-bmi_nond.mean())/pooled_std
print(effect_size)

**Observation (T-test: BMI vs Diabetes)**
- p-value approx 0 (p < 0.001) — we reject H0
- Conclusion: Mean BMI differs significantly between diabetic (mean=30.87) and non-diabetic (mean=27.15) patients
- Effect size (Cohen's d) approx 0.66, indicating a moderate difference
- Clinical interpretation: Higher BMI is statistically associated with diabetes status in this population

### Task 2 — Chi-Square: Exercise vs Diabetes

1) Question: Is there a significant association between exercise participation and diabetes status?

2) H0: There is no significant association between exercise participation and diabetes status.

   H1: There is a significant association between exercise participation and diabetes status.

In [ ]:
ect = pd.crosstab(df["EXERANY2"],df["Diabetes_binary"],normalize=False)
print(stats.chi2_contingency(ect))

**Observation (Chi-Square: Exercise vs Diabetes)**
- p-value ≈ 0 (p < 0.001) — we reject H0
- Chi-square statistic : 25608.74
- Degress of freedom : 1
- Conclusion: Exercise participation and diabetes status are significantly associated
- Clinical interpretation: People who report no exercise show a different diabetes distribution than those who report exercise in this population

### Task 3 — Chi-Square: Depression vs Diabetes

1) Question: Is there a significant association between Depressive Disorder and diabetes status?

2) H0: There is no significant association between Depressive Disorder and diabetes status.

   H1: There is a significant association between Depressive Disorder and diabetes status.

In [ ]:
act = pd.crosstab(df["ADDEPEV2"],df["Diabetes_binary"],normalize=False)
print(stats.chi2_contingency(act))

**Observation (Chi-Square: Depression vs Diabetes)**
- p-value ≈ 0 (p < 0.001) — we reject H0
- Chi-Square Statistic: 13282.53
- Degrees of Freedom - 1
- Conclusion: Depressive disorder and diabetes status are significantly associated
- Clinical interpretation: The diabetes distribution differs between those with and without a history of depressive disorder in this population

### Task 4 — Confidence Intervals for BMI

In [ ]:
ci_diabetic = stats.t.interval(
    confidence=0.95,
    df = len(BMI_diabetic) -1,
    loc = BMI_diabetic.mean(),
    scale = stats.sem(BMI_diabetic)
)
print(ci_diabetic[0]/100, ci_diabetic[1]/100)

In [ ]:
ci_non_diabetic = stats.t.interval (
    confidence = 0.95,
    df = len(BMI_nondiabetic) -1,
    loc = BMI_nondiabetic.mean(),
    scale = stats.sem(BMI_nondiabetic)
)
print(ci_non_diabetic[0]/100,ci_non_diabetic[1]/100)

**Observation (Task 4: BMI Confidence Intervals)**
- Diabetic CI (divided by 100): 30.8494 to 30.8961
- Non-diabetic CI (divided by 100): 27.1438 to 27.1597
- Overlap: No, these intervals do not overlap.
- Non-overlap implies the mean BMI is statistically different between the two groups at the 95% level.
- The intervals are extremely narrow because the sample size is very large, so the standard error $SE = SD / \sqrt{n}$ is tiny.

#  Probability Distributions & Bayes Theorem

### Task 1 — Identify Distributions in the Data

**Goal:** Diagnose which probability distributions best describe key variables before modeling.

**Plan:**
- Run a normality test on BMI to validate the shape assumption.
- Visualize BMI and PHYSHLTH to see skew, tails, and zero inflation.
- Map each variable to the closest theoretical distribution for interpretation.

In [ ]:
stat, p_value = stats.normaltest(df["_BMI5"]/100)
print(f"p-value for normality test: {p_value}")
plt.hist(df["_BMI5"],bins=1000,edgecolor="#BC2B2B")
plt.xlabel("BMI of the Interviewee")
plt.ylabel("Number of values")
plt.title("BMI Distribution")
plt.savefig("../reports/statistical/bmidist.png",bbox_inches= 'tight',dpi=150)
plt.show()

In [ ]:
pstat, pp_value = stats.normaltest(df["PHYSHLTH"]/100)
print(f"p-value for normality test: {p_value}")
plt.hist(df["PHYSHLTH"],bins=5,edgecolor="#000000")
plt.xlabel("Number of days Physical health wasnt good")
plt.ylabel("Number of values")
plt.title("Physical Health Distribution")
plt.savefig("../reports/statistical/physhlthdist.png",bbox_inches= 'tight',dpi=150)
plt.show()

#### Normality Test Results
- BMI normality test: p ≈ 0 → NOT normally distributed (log-normal)
- PHYSHLTH: Poisson distribution (count of rare events in fixed interval)
- Diabetes_binary: Bernoulli distribution (p ≈ 0.16)

#### Central Limit Theorem Application
Despite BMI not being normally distributed, the t-test results 
from Day 6 remain valid. By CLT, with n > 2M rows, the sampling 
distribution of the mean is approximately normal regardless of 
the underlying population distribution. CLT kicks in reliably 
at n > 30 — we have 2 million+.

### Task 2 — Bayes Theorem Application

**Question:** Given a high BMI, what is the probability of diabetes?

**Bayes Theorem:**
$P(\text{Diabetes} \mid \text{High BMI}) = \dfrac{P(\text{High BMI} \mid \text{Diabetes}) \times P(\text{Diabetes})}{P(\text{High BMI})}$

**Components (from the dataset):**
- $P(\text{Diabetes})$ = prior base rate (~14.85%)
- $P(\text{High BMI} \mid \text{Diabetes})$ = likelihood (high BMI among diabetics)
- $P(\text{High BMI})$ = evidence (high BMI overall)

In [ ]:
BMI_diabetic_df = pd.DataFrame(BMI_diabetic)
BMI_nondiabetic_df = pd.DataFrame(BMI_nondiabetic)

In [ ]:
p_diabetes = df["Diabetes_binary"].mean()
p_highbmi_diabetes = BMI_diabetic_df[BMI_diabetic_df["_BMI5"]>3000].count()/BMI_diabetic_df["_BMI5"].count()
p_highbmi = df[df["_BMI5"]>3000]["_BMI5"].count()/df["_BMI5"].count()
p_diabetes_highbmi = (p_highbmi_diabetes * p_diabetes)/p_highbmi

In [ ]:
print(p_diabetes)
print(p_highbmi_diabetes)
print(p_highbmi)
print(p_diabetes_highbmi)

**Observation (Task 2: Bayes Theorem)**
- $P(\text{Diabetes})$ = 14.85% in this dataset (prior).
- $P(\text{High BMI} \mid \text{Diabetes})$ = 47.91%
- $P(\text{High BMI})$ = 27.20%
- $P(\text{Diabetes} \mid \text{High BMI})$ = 26.16%, which is notably higher than the prior.
- Interpretation: Knowing a patient has BMI > 30 nearly doubles 
their diabetes probability from 14.85% to 26.17%. This quantifies 
the clinical value of BMI as a risk factor.
- This is an association-based estimate, not a causal claim.

### Extension to Naive Bayes
Adding multiple features (High BMI + No Exercise) extends Bayes 
by multiplying likelihoods under the independence assumption:
P(Diabetes | BMI, Exercise) ∝ P(BMI|D) × P(Exercise|D) × P(D)
This is the foundation of the Naive Bayes classifier in Week 4.

## Documentation for Statistical Analysis (Day 6 + Day 7)

1) T-test + effect size (BMI vs Diabetes)
    - Verified a statistically significant mean BMI difference between diabetic and non-diabetic groups (p < 0.001)
    - Computed Cohen's d to quantify effect size (moderate separation)

2) Chi-square tests (categorical associations)
    - Exercise vs Diabetes and Depression vs Diabetes both showed significant association (p < 0.001)
    - Interpreted results in the clinical context (behavioral and mental health links)

3) Confidence intervals for BMI
    - Built 95% CIs for mean BMI in both groups
    - Non-overlapping intervals confirmed meaningful group separation at scale

4) Probability distributions
    - Tested BMI and PHYSHLTH normality and visualized their distributions
    - Mapped BMI to log-normal, PHYSHLTH to Poisson, and Diabetes_binary to Bernoulli
    - Applied CLT reasoning to justify large-sample inference

5) Bayes Theorem application
    - Computed $P(\text{Diabetes} \mid \text{High BMI})$ using dataset priors and likelihoods
    - Compared posterior vs prior to show increased diabetes risk with BMI > 30
    - Added a Naive Bayes extension note for Week 4 modeling